# 3-D Benchmark — Prompt U-Net (multi-mode) vs nnInteractive

**Combined runner + results notebook.**

All P-UNet **modes** are evaluated on the **exact same initial prompt** per run.  
nnInteractive is run:
- once as a **baseline** (initial prompt only, no IFL)
- once **paired** with every IFL-enabled P-UNet mode, using the *same* `user_interacts_idx`

```
Example: modes = ['ssf', 'ifl', 'ifl_ssf']
  P-UNet ssf     —  compare with nn_baseline   (n_inter=0)
  P-UNet ifl     —  compare with nn+ifl         (n_inter = same as IFL produced)
  P-UNet ifl_ssf —  compare with nn+ifl_ssf     (n_inter = same as IFL+SSF produced)
```

| Workflow |
|---|
| **§1** Configure paths & parameters |
| **§2** Run the benchmark *(skip if loading saved results)* |
| **§3** Load results & analyse |
| **§4** Export flat CSV |

---
## §1 — Configuration

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

P_UNET_MODELS = [
    os.path.join(PROJECT_ROOT, 'training', 'p_unet_315.keras'),
]
NN_MODEL_DIR = None

NPZ_PATHS = [
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_ct.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_mri.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'SegRap2023.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'HCCTase_ceCT.npz'),
]

# Modes: all evaluated on the SAME initial prompt per run
# Each IFL mode => one extra paired nnInteractive run
MODES = ['ssf', 'ifl', 'ifl_ssf']

RUNS_PER_VOL      = 5
MODALITY          = None
OUTPUT_THRESHOLD  = 0.5
SSF_THRESHOLD     = 0.40
GT_DICE_THRESHOLD = 0.65
BATCH_SIZE        = 6
WINDOW            = 10
MIN_PROMPT_PIXELS = 50
NN_DEVICE         = None
MAX_VOLUMES       = None
OUTPUT_DIR        = os.getcwd()

print('Config OK')
print(f'  P-UNet modes : {MODES}')
print(f'  nnInteractive: baseline + one paired run per IFL mode')
print(f'  Datasets     : {[os.path.basename(p) for p in NPZ_PATHS]}')

---
## §2 — Run Benchmark

> Skip if you already have a `.pkl` file.

In [ ]:
from inference.ssf import RelativeSSIMStrategy
from evaluation.benchmark_nninteractive.benchmark_3d import run_benchmark

all_records = []
for p_model in P_UNET_MODELS:
    print(f'\n--- {os.path.basename(p_model)} ---')
    records = run_benchmark(
        npz_paths         = NPZ_PATHS,
        p_unet_model      = p_model,
        nn_model_dir      = NN_MODEL_DIR,
        runs_per_vol      = RUNS_PER_VOL,
        modes             = MODES,
        modality          = MODALITY,
        output_threshold  = OUTPUT_THRESHOLD,
        ssf_strategy      = RelativeSSIMStrategy(SSF_THRESHOLD),
        gt_dice_threshold = GT_DICE_THRESHOLD,
        batch_size        = BATCH_SIZE,
        window            = WINDOW,
        min_prompt_pixels = MIN_PROMPT_PIXELS,
        max_volumes       = MAX_VOLUMES,
        output_dir        = OUTPUT_DIR,
        nn_device         = NN_DEVICE,
        verbose           = True,
        return_predictions= False,
    )
    all_records.extend(records)

print(f'\nDone — {len(all_records)} runs collected.')

---
## §3 — Load & Analyse Results

In [ ]:
import pickle, glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Option A: use results already in memory
# records = all_records

# Option B: load newest pkl
pkl_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'results_*.pkl')))
if not pkl_files:
    raise FileNotFoundError(f'No results .pkl in {OUTPUT_DIR}')
LOAD_PATH = pkl_files[-1]
print(f'Loading: {LOAD_PATH}')
with open(LOAD_PATH, 'rb') as f:
    records = pickle.load(f)

MODES_EVALUATED = records[0]['modes_evaluated']
NN_KEYS         = sorted({k for r in records for k in r.get('nn_results', {})})
DATASETS        = sorted(set(r['dataset_name'] for r in records))
IFL_MODES       = [m for m in MODES_EVALUATED if 'ifl' in m]

print(f'Loaded {len(records)} runs')
print(f'P-UNet modes : {MODES_EVALUATED}')
print(f'nn_results   : {NN_KEYS}  (baseline + paired with {IFL_MODES})')
print(f'Datasets     : {DATASETS}')

In [ ]:
rows_long = []
for r in records:
    base = {
        'volume_id'   : r['volume_id'],
        'pid'         : r['pid'],
        'run_idx'     : r['run_idx'],
        'dataset_name': r['dataset_name'],
        'modality'    : r['modality'],
        'prompt_axis' : r['prompt_axis'],
    }
    for mode, md in r.get('per_mode', {}).items():
        paired = mode if 'ifl' in mode else 'baseline'
        rows_long.append({
            **base,
            'model'             : f'P-UNet [{mode}]',
            'mode'              : mode,
            'is_nn'             : False,
            'paired_nn_key'     : paired,
            'vol_dice'          : md['vol_dice'],
            'window_dice'       : md['window_dice'],
            'time_s'            : md['time_s'],
            'num_user_interacts': md.get('num_user_interacts'),
            'num_slices'        : md.get('num_slices_evaluated'),
            'num_interactions'  : None,
        })
    for nn_key, nn_md in r.get('nn_results', {}).items():
        label = 'nn_baseline' if nn_key == 'baseline' else f'nn+{nn_key}'
        rows_long.append({
            **base,
            'model'             : label,
            'mode'              : nn_key,
            'is_nn'             : True,
            'paired_nn_key'     : nn_key,
            'vol_dice'          : nn_md['vol_dice'],
            'window_dice'       : nn_md['window_dice'],
            'time_s'            : nn_md['time_s'],
            'num_user_interacts': None,
            'num_slices'        : None,
            'num_interactions'  : nn_md.get('num_interactions', 0),
        })

df_long     = pd.DataFrame(rows_long)
df_punet    = df_long[~df_long['is_nn']].copy()
df_nn       = df_long[df_long['is_nn']].copy()

PUNET_LABELS = [f'P-UNet [{m}]' for m in MODES_EVALUATED]
NN_LABELS    = ['nn_baseline'] + [f'nn+{m}' for m in IFL_MODES]
ALL_LABELS   = PUNET_LABELS + NN_LABELS

PALETTE = {
    'P-UNet [ssf]'    : '#4A90D9',
    'P-UNet [ifl]'    : '#50C878',
    'P-UNet [ifl_ssf]': '#9B59B6',
    'P-UNet [none]'   : '#BDC3C7',
    'nn_baseline'     : '#E87040',
    'nn+ifl'          : '#D4550F',
    'nn+ifl_ssf'      : '#A03010',
}
DEFAULT_COLOR = '#888888'

print(f'Long df: {len(df_long)} rows')
print(f'P-UNet labels: {PUNET_LABELS}')
print(f'NN labels    : {NN_LABELS}')
df_long.head()

### 3.1 — Overall summary

In [ ]:
def _fmt(s):
    s = pd.Series(s).dropna()
    return (
        f'{s.mean():.3f} ± {s.std():.3f}  '
        f'[med={s.median():.3f}  min={s.min():.3f}  max={s.max():.3f}  n={len(s)}]'
        if len(s) else 'N/A'
    )

print(f"{'='*78}")
print(f"  SUMMARY   n_runs={len(records)}")
print(f"{'='*78}")

for label in ALL_LABELS:
    sub = df_long[df_long['model'] == label]
    if sub.empty:
        continue
    print(f"\n  {label}")
    print(f"    Vol Dice    : {_fmt(sub['vol_dice'])}")
    print(f"    Window Dice : {_fmt(sub['window_dice'])}")
    print(f"    Time (s)    : {_fmt(sub['time_s'])}")
    if not sub['num_user_interacts'].dropna().empty:
        print(f"    IFL interacts: {_fmt(sub['num_user_interacts'])}")
    if not sub['num_interactions'].dropna().empty:
        print(f"    nn interactions: {_fmt(sub['num_interactions'])}")

print(f"\n{'='*78}")

### 3.2 — Boxplots: all models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(max(10, len(ALL_LABELS)*1.6), 5), sharey=True)
colors    = [PALETTE.get(l, DEFAULT_COLOR) for l in ALL_LABELS]

for metric, ax, title in [
    ('vol_dice',    axes[0], 'Volumetric Dice'),
    ('window_dice', axes[1], f'Window Dice (±{WINDOW} slices)'),
]:
    data   = [df_long[df_long['model'] == l][metric].dropna().tolist() for l in ALL_LABELS]
    xticks = [l.replace('P-UNet [', 'P-U\n[').replace(']', ']') for l in ALL_LABELS]
    bp = ax.boxplot(
        data, tick_labels=xticks,
        widths=0.52, patch_artist=True,
        boxprops=dict(linewidth=1.2),
        medianprops=dict(color='white', linewidth=2.2),
        whiskerprops=dict(linewidth=1),
        capprops=dict(linewidth=1),
        flierprops=dict(marker='o', markersize=3, alpha=0.5),
    )
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
    ax.axvline(len(PUNET_LABELS) + 0.5, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Dice')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

fig.suptitle(
    f'All Models — {len(records)} runs  |  same prompt per run',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_boxplots_all.pdf'), bbox_inches='tight')
plt.show()

### 3.3 — Paired scatter: P-UNet vs its nnInteractive counterpart

Each subplot shows the *fair* comparison:
- `ssf` vs `nn_baseline` (same initial prompt, no IFL)
- `ifl` vs `nn+ifl` (same interactions)
- `ifl_ssf` vs `nn+ifl_ssf` (same interactions)

In [ ]:
n_modes = len(MODES_EVALUATED)
fig, axes = plt.subplots(1, n_modes, figsize=(5 * n_modes, 5))
if n_modes == 1:
    axes = [axes]

for ax, mode in zip(axes, MODES_EVALUATED):
    paired_key = mode if 'ifl' in mode else 'baseline'
    nn_label   = 'nn_baseline' if paired_key == 'baseline' else f'nn+{paired_key}'

    p_sub  = df_long[df_long['model'] == f'P-UNet [{mode}]'][['volume_id','run_idx','vol_dice']]
    nn_sub = df_long[df_long['model'] == nn_label][['volume_id','run_idx','vol_dice']]

    merged = p_sub.merge(nn_sub, on=['volume_id','run_idx'], suffixes=('_punet','_nn'))

    color = PALETTE.get(f'P-UNet [{mode}]', DEFAULT_COLOR)
    ax.scatter(merged['vol_dice_punet'], merged['vol_dice_nn'],
               alpha=0.6, s=28, color=color, edgecolors='none')
    ax.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.5)

    delta = (merged['vol_dice_punet'] - merged['vol_dice_nn']).mean()
    ax.set_title(
        f'P-UNet [{mode}]\nvs {nn_label}\n'
        f'Δ Dice (P-UNet − nn) = {delta:+.3f}',
        fontsize=10
    )
    ax.set_xlabel(f'P-UNet [{mode}]', fontsize=9)
    ax.set_ylabel(nn_label, fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.25)

fig.suptitle('Paired Scatter — P-UNet vs nnInteractive (same prompt & interactions)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scatter_paired.pdf'), bbox_inches='tight')
plt.show()

### 3.4 — Per-dataset summary table

In [ ]:
rows = []
for ds in DATASETS:
    ds_df = df_long[df_long['dataset_name'] == ds]
    for label in ALL_LABELS:
        sub = ds_df[ds_df['model'] == label]
        if sub.empty:
            continue
        rows.append({
            'dataset'  : ds,
            'model'    : label,
            'n_runs'   : len(sub),
            'vol_mean' : sub['vol_dice'].mean(),
            'vol_std'  : sub['vol_dice'].std(),
            'win_mean' : sub['window_dice'].mean(),
            'win_std'  : sub['window_dice'].std(),
            'time_mean': sub['time_s'].mean(),
        })

ds_table = pd.DataFrame(rows)
ds_table['Vol Dice'] = ds_table.apply(lambda r: f"{r['vol_mean']:.3f} ±{r['vol_std']:.3f}", axis=1)
ds_table['Win Dice'] = ds_table.apply(lambda r: f"{r['win_mean']:.3f} ±{r['win_std']:.3f}", axis=1)
ds_table['Time (s)'] = ds_table['time_mean'].map(lambda x: f'{x:.1f}s')
display(ds_table[['dataset','model','n_runs','Vol Dice','Win Dice','Time (s)']].set_index(['dataset','model']))

### 3.5 — Per-dataset bar charts

In [ ]:
n_ds = len(DATASETS)
fig, axes = plt.subplots(n_ds, 2, figsize=(13, 4.2 * n_ds), squeeze=False)
x      = np.arange(len(ALL_LABELS))
bcolors = [PALETTE.get(l, DEFAULT_COLOR) for l in ALL_LABELS]

for row, ds in enumerate(DATASETS):
    ds_df = df_long[df_long['dataset_name'] == ds]
    for col, (metric, title) in enumerate([
        ('vol_dice',    'Volumetric Dice'),
        ('window_dice', f'Window Dice (±{WINDOW})')]
    ):
        ax    = axes[row][col]
        means = [ds_df[ds_df['model'] == l][metric].mean() for l in ALL_LABELS]
        stds  = [ds_df[ds_df['model'] == l][metric].std()  for l in ALL_LABELS]
        bars  = ax.bar(x, means, 0.62, yerr=stds, color=bcolors,
                       edgecolor='white', lw=0.8, capsize=4)
        ax.axvline(len(PUNET_LABELS) - 0.5, color='grey', lw=0.8, ls='--', alpha=0.5)
        ax.set_xticks(x)
        xtick_labels = [l.replace('P-UNet [','P-U\n[') for l in ALL_LABELS]
        ax.set_xticklabels(xtick_labels, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_title(f'{ds}  —  {title}', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        for bar, mean in zip(bars, means):
            if not np.isnan(mean):
                ax.text(bar.get_x()+bar.get_width()/2, mean+0.02,
                        f'{mean:.3f}', ha='center', va='bottom', fontsize=7)
        if col == 0:
            ax.set_ylabel(f'{ds}\nDice', fontsize=9)

fig.suptitle('Per-Dataset Dice by Model', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_per_dataset_bars.pdf'), bbox_inches='tight')
plt.show()

### 3.6 — Per-dataset Dice heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(max(11, len(ALL_LABELS)*1.6), max(4, len(DATASETS)*0.9)))

for ax, (metric, title) in zip(axes, [
    ('vol_dice',    'Volumetric Dice'),
    ('window_dice', f'Window Dice (±{WINDOW})'),
]):
    matrix = np.array([
        [df_long[(df_long['dataset_name']==ds)&(df_long['model']==l)][metric].mean()
         for l in ALL_LABELS]
        for ds in DATASETS
    ])
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(ALL_LABELS)))
    ax.set_xticklabels([l.replace('P-UNet [','P-U\n[') for l in ALL_LABELS], fontsize=8)
    ax.set_yticks(range(len(DATASETS)))
    ax.set_yticklabels(DATASETS, fontsize=9)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax)
    ax.axvline(len(PUNET_LABELS) - 0.5, color='white', lw=1.5, ls='--')
    for i in range(len(DATASETS)):
        for j in range(len(ALL_LABELS)):
            v = matrix[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                        color='black' if 0.2<v<0.8 else 'white', fontsize=8)

fig.suptitle('Per-Dataset Dice Heatmaps', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_heatmap_per_dataset.pdf'), bbox_inches='tight')
plt.show()

### 3.7 — Inference timing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax   = axes[0]
x    = np.arange(len(ALL_LABELS))
means= [df_long[df_long['model']==l]['time_s'].mean() for l in ALL_LABELS]
stds = [df_long[df_long['model']==l]['time_s'].std()  for l in ALL_LABELS]
bars = ax.bar(x, means, 0.6, yerr=stds, color=[PALETTE.get(l,DEFAULT_COLOR) for l in ALL_LABELS],
              edgecolor='white', lw=0.8, capsize=4)
ax.axvline(len(PUNET_LABELS)-0.5, color='grey', lw=0.8, ls='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([l.replace('P-UNet [','P-U\n[') for l in ALL_LABELS], fontsize=8)
ax.set_ylabel('Inference Time (s)')
ax.set_title('Average Inference Time by Model', fontsize=11)
ax.grid(axis='y', alpha=0.3)
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x()+bar.get_width()/2, mean+std+0.3,
            f'{mean:.1f}s', ha='center', va='bottom', fontsize=7.5)

ax  = axes[1]
tm  = np.array([
    [df_long[(df_long['dataset_name']==ds)&(df_long['model']==l)]['time_s'].mean()
     for l in ALL_LABELS]
    for ds in DATASETS
])
im  = ax.imshow(tm, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(ALL_LABELS)))
ax.set_xticklabels([l.replace('P-UNet [','P-U\n[') for l in ALL_LABELS], fontsize=8)
ax.set_yticks(range(len(DATASETS)))
ax.set_yticklabels(DATASETS, fontsize=9)
ax.set_title('Inference Time Heatmap (s)', fontsize=11)
ax.axvline(len(PUNET_LABELS)-0.5, color='white', lw=1.5, ls='--')
plt.colorbar(im, ax=ax, label='seconds')
vmax = np.nanmax(tm)
for i in range(len(DATASETS)):
    for j in range(len(ALL_LABELS)):
        v = tm[i,j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                    color='black' if v<0.7*vmax else 'white', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'timing_analysis.pdf'), bbox_inches='tight')
plt.show()

### 3.8 — IFL interaction budget: P-UNet vs nnInteractive

In [ ]:
if not IFL_MODES:
    print('No IFL modes in this benchmark.')
else:
    fig, axes = plt.subplots(2, len(IFL_MODES),
                             figsize=(5.5*len(IFL_MODES), 7), squeeze=False)

    for col, mode in enumerate(IFL_MODES):
        nn_label = f'nn+{mode}'

        ax0   = axes[0][col]
        punet_counts = df_long[(df_long['model']==f'P-UNet [{mode}]') &
                                df_long['num_user_interacts'].notna()]['num_user_interacts'].astype(int)
        color_p = PALETTE.get(f'P-UNet [{mode}]', DEFAULT_COLOR)
        if not punet_counts.empty:
            bins = range(punet_counts.min(), punet_counts.max()+2)
            ax0.hist(punet_counts, bins=bins, align='left', rwidth=0.72,
                     color=color_p, edgecolor='white')
        ax0.set_title(f'P-UNet [{mode}]\nn_interacts (incl. initial)', fontsize=10)
        ax0.set_xlabel('Interactions')
        ax0.set_ylabel('Runs')
        ax0.grid(axis='y', alpha=0.3)
        if not punet_counts.empty:
            ax0.axvline(punet_counts.mean(), color='red', lw=1.5,
                        linestyle='--', label=f'mean={punet_counts.mean():.1f}')
            ax0.legend(fontsize=8)

        ax1   = axes[1][col]
        nn_counts = df_long[(df_long['model']==nn_label) &
                             df_long['num_interactions'].notna()]['num_interactions'].astype(int)
        color_n = PALETTE.get(nn_label, DEFAULT_COLOR)
        if not nn_counts.empty:
            bins = range(nn_counts.min(), nn_counts.max()+2)
            ax1.hist(nn_counts, bins=bins, align='left', rwidth=0.72,
                     color=color_n, edgecolor='white')
        ax1.set_title(f'{nn_label}\nn_interactions given', fontsize=10)
        ax1.set_xlabel('Interactions (extra slices beyond initial prompt)')
        ax1.set_ylabel('Runs')
        ax1.grid(axis='y', alpha=0.3)

    fig.suptitle('IFL Interaction Budget — P-UNet vs nnInteractive', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ifl_interactions.pdf'), bbox_inches='tight')
    plt.show()

### 3.9 — Dice by prompt axis

In [ ]:
axis_labels  = {0: 'Axial', 1: 'Coronal', 2: 'Sagittal'}
present_axes = sorted(df_long['prompt_axis'].dropna().unique().astype(int))

fig, axes_row = plt.subplots(1, len(ALL_LABELS),
                             figsize=(3.5*len(ALL_LABELS), 4.5), sharey=True)
if len(ALL_LABELS) == 1:
    axes_row = [axes_row]

for ax, label in zip(axes_row, ALL_LABELS):
    sub     = df_long[df_long['model'] == label]
    grouped = [sub[sub['prompt_axis']==a]['vol_dice'].dropna().tolist() for a in present_axes]
    bp = ax.boxplot(
        grouped,
        tick_labels=[axis_labels.get(a, str(a)) for a in present_axes],
        patch_artist=True, widths=0.5,
        boxprops=dict(facecolor=PALETTE.get(label, DEFAULT_COLOR), linewidth=1.2),
        medianprops=dict(color='white', linewidth=2),
        whiskerprops=dict(linewidth=1), capprops=dict(linewidth=1),
    )
    short = label.replace('P-UNet [', 'P-U[').replace(']', ']')
    ax.set_title(short, fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Vol Dice')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Volumetric Dice by Prompt Axis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_by_prompt_axis.pdf'), bbox_inches='tight')
plt.show()

---
### 3.10 — Failure case visualization

Reloads the raw volume from disk for every low-Dice run and shows slice context  
around the initial prompt so you can understand *why* the model failed.

| Row | Content |
|-----|---------|
| 0   | Raw image (grayscale) |
| 1   | Image + GT overlay (green) |
| 2   | Image + GT (green) + prompt mask (yellow, prompt slice only) |

The **★ PROMPT** column is framed in gold.  
The figure title shows Dice for every evaluated mode + paired nnInteractive.

In [ ]:
# ── Failure-case config ───────────────────────────────────────────────────────
FAILURE_THRESHOLD = 0.30   # vol_dice below this = 'failure'
TOP_N_FAILURES    = 8      # max cases shown
MODE_TO_RANK_BY   = MODES_EVALUATED[0]  # rank by this mode's vol_dice
CONTEXT_SLICES    = 3      # ±slices around prompt slice
SAVE_FAILURE_FIGS = True   # save one PDF per case

# Map dataset_name → .npz path  (NPZ_PATHS from §1)
from pathlib import Path as _PF
try:
    DS_TO_PATH = {_PF(p).stem: p for p in NPZ_PATHS}
except NameError:
    DS_TO_PATH = {}   # NPZ_PATHS not in scope — fill DS_TO_PATH manually
    print('WARNING: NPZ_PATHS not found. Run §1 first or set DS_TO_PATH manually.')

print(f'Ranking mode        : [{MODE_TO_RANK_BY}]')
print(f'Failure threshold   : vol_dice < {FAILURE_THRESHOLD}')
print(f'Dataset map         : {list(DS_TO_PATH.keys())}')

In [ ]:
from data.test_data.ds_handler import load_dataset as _load_ds


def _pct_norm(arr):
    lo, hi = np.percentile(arr, [2, 98])
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def _mask_rgba(mask, rgb, alpha):
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[mask > 0] = [*rgb, alpha]
    return rgba


def _all_dice_str(rec):
    parts = []
    for m in rec.get('modes_evaluated', []):
        d = rec.get('per_mode', {}).get(m, {}).get('vol_dice', float('nan'))
        parts.append(f'[P-UNet {m}] {d:.3f}')
    for nk, nd in rec.get('nn_results', {}).items():
        lbl = 'nn_base' if nk == 'baseline' else f'nn+{nk}'
        parts.append(f'[{lbl}] {nd.get("vol_dice", float("nan")):.3f}')
    return '   '.join(parts)


# ── Find failure records ──────────────────────────────────────────────────────
fail_df = (
    df_long[
        (df_long['mode'] == MODE_TO_RANK_BY) &
        (df_long['vol_dice'] < FAILURE_THRESHOLD)
    ]
    .copy()
    .sort_values('vol_dice')
    .drop_duplicates(subset=['volume_id', 'run_idx'])
    .head(TOP_N_FAILURES)
)

if fail_df.empty:
    print(f'No failures found (vol_dice < {FAILURE_THRESHOLD} for [{MODE_TO_RANK_BY}]).')
    print('Try raising FAILURE_THRESHOLD or choosing a different MODE_TO_RANK_BY.')

# ── Dataset cache ─────────────────────────────────────────────────────────────
_ds_cache = {}
_ax_name  = {0: 'Axial', 1: 'Coronal', 2: 'Sagittal'}


for _, frow in fail_df.iterrows():
    vid      = frow['volume_id']
    pid      = frow['pid']
    ds_name  = frow['dataset_name']

    rec = next(
        (r for r in records
         if r['volume_id'] == vid and r['run_idx'] == frow['run_idx']),
        None,
    )
    if rec is None:
        print(f'  [SKIP] record not found for {vid}')
        continue

    p_axis  = int(rec['prompt_axis'])
    p_idx   = int(rec['prompt_idx'])
    sel_roi = int(rec['selected_roi'])

    # Load volume
    npz_path = DS_TO_PATH.get(ds_name)
    if not npz_path:
        print(f'  [SKIP] no .npz mapped for "{ds_name}"')
        continue
    if ds_name not in _ds_cache:
        try:
            _ds_cache[ds_name] = _load_ds(npz_path)
        except Exception as e:
            print(f'  [SKIP] load error: {e}')
            continue
    dataset = _ds_cache[ds_name]
    if pid not in dataset:
        print(f'  [SKIP] pid "{pid}" not in {ds_name}')
        continue

    item   = dataset[pid]
    img_3d = np.asarray(item['image']).astype(np.float32)
    segs   = item['segmentations']
    if isinstance(segs, list):
        seg_3d = np.zeros_like(img_3d, dtype=np.int32)
        for li, s in enumerate(segs, 1):
            seg_3d[np.asarray(s) != 0] = li
    else:
        seg_3d = np.asarray(segs).astype(np.int32)
    seg_bin = (seg_3d == sel_roi).astype(np.float32)

    # Slice range
    n_ax     = img_3d.shape[p_axis]
    show_idx = list(range(max(0, p_idx - CONTEXT_SLICES),
                          min(n_ax, p_idx + CONTEXT_SLICES + 1)))
    n_cols   = len(show_idx)

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, axg = plt.subplots(
        3, n_cols,
        figsize=(3.0 * n_cols, 2.9 * 3),
        gridspec_kw={'hspace': 0.03, 'wspace': 0.03},
    )
    if n_cols == 1:
        axg = axg.reshape(3, 1)

    # GT voxel count for context
    roi_vox = int(seg_bin.sum())
    fail_d  = rec['per_mode'].get(MODE_TO_RANK_BY, {}).get('vol_dice', float('nan'))
    fig.suptitle(
        f'FAILURE  [{MODE_TO_RANK_BY}] vol_dice={fail_d:.3f}  │  '
        f'{ds_name} / {pid} / ROI={sel_roi} ({roi_vox} vox)\n'
        f'Prompt: {_ax_name.get(p_axis,"?")} axis · slice {p_idx} '
        f'(run {rec["run_idx"]})\n'
        f'{_all_dice_str(rec)}',
        fontsize=8.5, y=1.015, color='white',
    )

    ROW_LABELS = [
        'Image',
        'Image + GT',
        'Image + GT\n+ Prompt (★)',
    ]

    for col, si in enumerate(show_idx):
        img_sl = np.take(img_3d,  si, axis=p_axis)
        seg_sl = np.take(seg_bin, si, axis=p_axis)
        img_d  = _pct_norm(img_sl)
        is_pmt = (si == p_idx)

        for row in range(3):
            ax = axg[row, col]
            ax.imshow(img_d, cmap='gray', vmin=0, vmax=1, interpolation='bilinear')

            # Row 1 & 2: GT overlay (green)
            if row >= 1 and seg_sl.any():
                ax.imshow(_mask_rgba(seg_sl, [0.18, 0.88, 0.32], 0.42),
                          interpolation='nearest')

            # Row 2, prompt slice only: prompt mask (yellow on top)
            if row == 2 and is_pmt and seg_sl.any():
                ax.imshow(_mask_rgba(seg_sl, [1.0, 0.88, 0.1], 0.62),
                          interpolation='nearest')

            ax.axis('off')

            # Left row label
            if col == 0:
                ax.set_ylabel(ROW_LABELS[row], fontsize=7.5, color='#cccccc',
                              rotation=90, labelpad=4, va='center')
                ax.yaxis.set_visible(True)
                ax.set_yticks([])

            # Top column title
            if row == 0:
                ax.set_title(
                    f'{_ax_name.get(p_axis,"")} {si}' + ('\n★ PROMPT' if is_pmt else ''),
                    fontsize=7.5,
                    color='gold' if is_pmt else '#aaaaaa',
                )

            # Spine
            for sp in ax.spines.values():
                if is_pmt:
                    sp.set_visible(True)
                    sp.set_edgecolor('gold')
                    sp.set_linewidth(2.2)
                else:
                    sp.set_visible(False)

    fig.patch.set_facecolor('#181818')
    for ax in axg.flat:
        ax.set_facecolor('#181818')

    if SAVE_FAILURE_FIGS:
        safe = vid.replace('/', '_').replace('\\', '_')
        pdf  = os.path.join(OUTPUT_DIR, f'failure_{safe}_run{rec["run_idx"]}.pdf')
        fig.savefig(pdf, bbox_inches='tight', facecolor='#181818')
        print(f'  → {pdf}')

    plt.show()
    plt.close(fig)

print('Failure visualization complete.')

---
## §4 — Export flat CSV

In [ ]:
csv_rows = []
for r in records:
    row = {
        'volume_id'    : r['volume_id'],
        'pid'          : r['pid'],
        'run_idx'      : r['run_idx'],
        'dataset_name' : r['dataset_name'],
        'modality'     : r['modality'],
        'p_unet_model' : r['p_unet_model'],
        'prompt_axis'  : r['prompt_axis'],
        'prompt_idx'   : r['prompt_idx'],
        'selected_roi' : r['selected_roi'],
    }
    for mode, md in r.get('per_mode', {}).items():
        row[f'{mode}_vol_dice']           = md['vol_dice']
        row[f'{mode}_window_dice']        = md['window_dice']
        row[f'{mode}_time_s']             = md['time_s']
        row[f'{mode}_num_slices']         = md.get('num_slices_evaluated')
        row[f'{mode}_num_user_interacts'] = md.get('num_user_interacts')
    for nn_key, nn_md in r.get('nn_results', {}).items():
        col_pref = 'nn_baseline' if nn_key == 'baseline' else f'nn_{nn_key}'
        row[f'{col_pref}_vol_dice']       = nn_md['vol_dice']
        row[f'{col_pref}_window_dice']    = nn_md['window_dice']
        row[f'{col_pref}_time_s']         = nn_md['time_s']
        row[f'{col_pref}_n_interactions'] = nn_md.get('num_interactions', 0)
    csv_rows.append(row)

df_export = pd.DataFrame(csv_rows)
csv_path  = os.path.join(OUTPUT_DIR, 'results_flat.csv')
df_export.to_csv(csv_path, index=False)
print(f'Exported {len(df_export)} rows → {csv_path}')
df_export.head()